Go through a list supernovae and templates,
for each template-sn fit splines
store if successful.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import json
import re
import pymongo
from warptemplate import get_ztftable_from_ampel, get_template_correction

In [ ]:
# References warpclasses 
classnbr = 2

In [ ]:
warpclasses = [
    'SN Ib', 'SN Ia-91bg', 'SN II', 'SN Ia-91T',
    'SN Ib/c', 'SN Ia-pec', 'SN IIP', 'SN Iax', 'SN Ic',
    'SLSN-II']

In [ ]:
fdir = '/Users/jnordin/data/models/sncosmo/'

In [ ]:
client = pymongo.MongoClient()
db = client.bts_ipacfp_strictbase    # Only final lc. try bts_ipacfp_strictbase_full for all alerts

In [ ]:
df = pd.read_csv( fdir+'warpmod/templatematches_'+re.sub(r'/', '', warpclasses[classnbr])+'.csv' )

In [ ]:
infocollect = {}

In [ ]:
df.shape

In [ ]:
for k, row in df.iterrows():
    name, z, model, sntype = row['id'], row['z'], row['model'], row['class']
    print(k, name, model)

    if re.search( 'salt', model ):
        print(' ... salt models not part of source.')
        continue 

    
    if not name in infocollect:
        infocollect[name] = {}

    tab = get_ztftable_from_ampel( name, db, redshift=z, type = sntype ) 

    
    mdict = get_template_correction( 
        tab, model, z, plot_dir = '/Users/jnordin/tmp/warptest', plot_label=name
    )
    if not mdict['success']:
        print('... did not work!!')
        continue
    # Retain the important keys:
    # - corrmodel: the input to WarpedTimeSeriesSource 
    infocollect[name][model] = {
        k:mdict[k] for k in 
        ['dps_init', 'hostebv', 'success', 'ndof', 'absmag', 'chidof', 'lceval', 'corrmodel']
    }


In [ ]:
storefile = '/Users/jnordin/tmp/warps_'+re.sub(r'/', '', warpclasses[classnbr])+'.pkl'

In [ ]:
with open(storefile, 'wb') as file:
    pickle.dump(infocollect, file)

In [ ]:
#with open(storefile, 'rb') as file:
#    foo = pickle.load(file)